# **SSA Document Collection Pipeline**
The notebook preps the Social Security Administration document datasets. Unlike other agencies, the SSA website applied access restrictions that prevented reliable automated crawling from the cloud execution environement. So after trying multiple approaches, a curated list of the SSA form URLs was used to build the dataset

## Initialize Project Environment

In [ ]:
import requests
from bs4 import BeautifulSoup
from pathlib import Path
from urllib.parse import urljoin
import pandas as pd
import re
import time

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


## Verify Existing Project Structure

In [ ]:
from pathlib import Path

project = Path("/content/drive/MyDrive/newstart_ai")

print("Project exists:", project.exists())

raw_path = project / "data/raw"
metadata_path = project / "data/metadata"

print("Raw exists:", raw_path.exists())
print("Metadata exists:", metadata_path.exists())

Project exists: True
Raw exists: True
Metadata exists: True


# Explore SSA Forms Website

The initial goal was the crawl the SSA Forms Website the same way I did with the first three agencies. This investigates the website and sees wether its possible from multiple avenues. For notebook organization sake, much of the exploration was deleted for brevity sake.

In [ ]:
url = "https://www.ssa.gov/forms/forms.html"

response = requests.get(
    url,
    headers=headers
)

print(response.status_code)
print(response.text[:500])

403
<HTML><HEAD>
<TITLE>Access Denied</TITLE>
</HEAD><BODY>
<H1>Access Denied</H1>
 
You don't have permission to access "http&#58;&#47;&#47;www&#46;ssa&#46;gov&#47;forms&#47;forms&#46;html" on this server.<P>
Reference&#32;&#35;18&#46;5c8a1402&#46;1783965569&#46;1c9d9222
<P>https&#58;&#47;&#47;errors&#46;edgesuite&#46;net&#47;18&#46;5c8a1402&#46;1783965569&#46;1c9d9222</P>
</BODY>
</HTML>



In [ ]:
import requests
from bs4 import BeautifulSoup
import random
import time


session = requests.Session()

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/120.0 Safari/537.36"
    ),
    "Accept": (
        "text/html,application/xhtml+xml,"
        "application/xml;q=0.9,image/webp,*/*;q=0.8"
    ),
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.google.com/"
}


url = "https://www.ssa.gov/forms/"


try:

    response = session.get(
        url,
        headers=headers,
        timeout=30
    )

    print("Status:", response.status_code)
    print(response.text[:500])


except Exception as e:
    print("Error:")
    print(e)

Status: 403
<HTML><HEAD>
<TITLE>Access Denied</TITLE>
</HEAD><BODY>
<H1>Access Denied</H1>
 
You don't have permission to access "http&#58;&#47;&#47;www&#46;ssa&#46;gov&#47;forms&#47;" on this server.<P>
Reference&#32;&#35;18&#46;ca071002&#46;1783965812&#46;13922ecf
<P>https&#58;&#47;&#47;errors&#46;edgesuite&#46;net&#47;18&#46;ca071002&#46;1783965812&#46;13922ecf</P>
</BODY>
</HTML>



## Alternative Data Collection Strategy

Testing showed that the SSA website restricts automated access to many of its indexing pages from cloud-hosted environments.
To maintain consistency with the other agency datasets, a curated list of publicly accessible SSA PDF form URLs was assembled. These documents were then processed using the same download, validation, and metadata generation pipeline used throughout the project. The following list contains publicly accessible SSA forms collected for this project. The forms span several categories, including retirement benefits, disability, SSI, Medicare, appeals, representative payee services, and Social Security card applications.

In [ ]:
ssa_pdf_urls = [

    # ----------------------------
    # Retirement / Benefits
    # ----------------------------
    "https://www.ssa.gov/forms/ssa-1.pdf",
    "https://www.ssa.gov/forms/ssa-2.pdf",
    "https://www.ssa.gov/forms/ssa-5.pdf",
    "https://www.ssa.gov/forms/ssa-10.pdf",
    "https://www.ssa.gov/forms/ssa-16.pdf",
    "https://www.ssa.gov/forms/ssa-18.pdf",
    "https://www.ssa.gov/forms/ssa-24.pdf",
    "https://www.ssa.gov/forms/ssa-25.pdf",
    "https://www.ssa.gov/forms/ssa-44.pdf",
    "https://www.ssa.gov/forms/ssa-55.pdf",
    "https://www.ssa.gov/forms/ssa-1-bk.pdf",
    "https://www.ssa.gov/forms/ssa-2-bk.pdf",
    "https://www.ssa.gov/forms/ssa-3.pdf",
    "https://www.ssa.gov/forms/ssa-4.pdf",
    "https://www.ssa.gov/forms/ssa-6.pdf",
    "https://www.ssa.gov/forms/ssa-7.pdf",
    "https://www.ssa.gov/forms/ssa-8.pdf",
    "https://www.ssa.gov/forms/ssa-9.pdf",
    "https://www.ssa.gov/forms/ssa-10-bk.pdf",
    "https://www.ssa.gov/forms/ssa-18-bk.pdf",


    # ----------------------------
    # Disability
    # ----------------------------
    "https://www.ssa.gov/forms/ssa-3368.pdf",
    "https://www.ssa.gov/forms/ssa-3369.pdf",
    "https://www.ssa.gov/forms/ssa-827.pdf",
    "https://www.ssa.gov/forms/ssa-454.pdf",
    "https://www.ssa.gov/forms/ssa-455.pdf",
    "https://www.ssa.gov/forms/ssa-454-bk.pdf",
    "https://www.ssa.gov/forms/ssa-3441.pdf",
    "https://www.ssa.gov/forms/ssa-3820.pdf",
    "https://www.ssa.gov/forms/ssa-3820-bk.pdf",
    "https://www.ssa.gov/forms/ssa-4734.pdf",
    "https://www.ssa.gov/forms/ssa-4734-f4.pdf",
    "https://www.ssa.gov/forms/ssa-827-bk.pdf",
    "https://www.ssa.gov/forms/ssa-795.pdf",
    "https://www.ssa.gov/forms/ssa-795-sp.pdf",
    "https://www.ssa.gov/forms/ssa-455-o.pdf",


    # ----------------------------
    # SSI
    # ----------------------------
    "https://www.ssa.gov/forms/ssa-8000.pdf",
    "https://www.ssa.gov/forms/ssa-8001.pdf",
    "https://www.ssa.gov/forms/ssa-8010.pdf",
    "https://www.ssa.gov/forms/ssa-8011.pdf",
    "https://www.ssa.gov/forms/ssa-821.pdf",
    "https://www.ssa.gov/forms/ssa-820.pdf",
    "https://www.ssa.gov/forms/ssa-8001-bk.pdf",
    "https://www.ssa.gov/forms/ssa-8010-bk.pdf",
    "https://www.ssa.gov/forms/ssa-8011-bk.pdf",
    "https://www.ssa.gov/forms/ssa-820-f4.pdf",
    "https://www.ssa.gov/forms/ssa-821-bk.pdf",
    "https://www.ssa.gov/forms/ssa-8510.pdf",
    "https://www.ssa.gov/forms/ssa-8510-f4.pdf",


    # ----------------------------
    # Medicare
    # ----------------------------
    "https://www.ssa.gov/forms/ssa-1020.pdf",
    "https://www.ssa.gov/forms/ssa-1021.pdf",
    "https://www.ssa.gov/forms/ssa-1020-internet.pdf",
    "https://www.ssa.gov/forms/ssa-795-op.pdf",
    "https://www.ssa.gov/forms/ssa-1020b.pdf",
    "https://www.ssa.gov/forms/ssa-1020-op.pdf",


    # ----------------------------
    # Social Security Card / Identity
    # ----------------------------
    "https://www.ssa.gov/forms/ss-5.pdf",
    "https://www.ssa.gov/forms/ss-5fs.pdf",
    "https://www.ssa.gov/forms/ss-5-sp.pdf",
    "https://www.ssa.gov/forms/ss-5-fs.pdf",


    # ----------------------------
    # Earnings / Records
    # ----------------------------
    "https://www.ssa.gov/forms/ssa-7050.pdf",
    "https://www.ssa.gov/forms/ssa-7004.pdf",
    "https://www.ssa.gov/forms/ssa-7005.pdf",
    "https://www.ssa.gov/forms/ssa-7050-fs.pdf",


    # ----------------------------
    # Representative Payee
    # ----------------------------
    "https://www.ssa.gov/forms/ssa-11-bk.pdf",
    "https://www.ssa.gov/forms/ssa-13.pdf",
    "https://www.ssa.gov/forms/ssa-14.pdf",
    "https://www.ssa.gov/forms/ssa-15.pdf",
    "https://www.ssa.gov/forms/ssa-787.pdf",
    "https://www.ssa.gov/forms/ssa-623.pdf",
    "https://www.ssa.gov/forms/ssa-6230.pdf",
    "https://www.ssa.gov/forms/ssa-6234.pdf",
    "https://www.ssa.gov/forms/ssa-11.pdf",


    # ----------------------------
    # Appeals / Reviews
    # ----------------------------
    "https://www.ssa.gov/forms/ssa-561-u2.pdf",
    "https://www.ssa.gov/forms/ssa-3441-bk.pdf",
    "https://www.ssa.gov/forms/ssa-789.pdf",
    "https://www.ssa.gov/forms/ssa-3441-ext.pdf",
    "https://www.ssa.gov/forms/ssa-561.pdf",


    # ----------------------------
    # Authorization / Consent
    # ----------------------------
    "https://www.ssa.gov/forms/ssa-827-op1.pdf",
    "https://www.ssa.gov/forms/ssa-827-op2.pdf",
    "https://www.ssa.gov/forms/ssa-3288.pdf",
    "https://www.ssa.gov/forms/ssa-3288-op.pdf",
    "https://www.ssa.gov/forms/ssa-7050-f4.pdf",
    "https://www.ssa.gov/forms/ssa-7004-sup.pdf",


    # ----------------------------
    # Miscellaneous
    # ----------------------------
    "https://www.ssa.gov/forms/ssa-2855.pdf",
    "https://www.ssa.gov/forms/ssa-2854.pdf",
    "https://www.ssa.gov/forms/ssa-2856.pdf",
    "https://www.ssa.gov/forms/ssa-2858.pdf",
    "https://www.ssa.gov/forms/ssa-2455.pdf",
    "https://www.ssa.gov/forms/ssa-2459.pdf",
    "https://www.ssa.gov/forms/ssa-2490.pdf",
    "https://www.ssa.gov/forms/ssa-2490-bk.pdf",
    "https://www.ssa.gov/forms/ssa-2490-op.pdf",
    "https://www.ssa.gov/forms/ssa-131.pdf",
    "https://www.ssa.gov/forms/ssa-1380.pdf",
    "https://www.ssa.gov/forms/ssa-1709.pdf",
    "https://www.ssa.gov/forms/ssa-1724.pdf",
]


In [ ]:
print("SSA candidate PDFs:", len(ssa_pdf_urls))

SSA candidate PDFs: 95


In [ ]:
import requests


valid_ssa_pdfs = []

headers = {
    "User-Agent": "Mozilla/5.0"
}

for url in ssa_pdf_urls:

    try:
        response = requests.get(
            url,
            headers=headers,
            timeout=20,
            stream=True
        )

        content_type = response.headers.get("Content-Type", "")

        if (
            response.status_code == 200
            and "application/pdf" in content_type.lower()
        ):
            valid_ssa_pdfs.append(url)
            print("VALID:", url.split("/")[-1])

        else:
            print(
                "FAILED:",
                response.status_code,
                url.split("/")[-1],
                content_type
            )

    except Exception as e:
        print(
            "ERROR:",
            url.split("/")[-1],
            e
        )


print("\n----------------------")
print("Valid SSA PDFs:", len(valid_ssa_pdfs))

FAILED: 403 ssa-1.pdf text/html
FAILED: 403 ssa-2.pdf text/html
FAILED: 403 ssa-5.pdf text/html
FAILED: 403 ssa-10.pdf text/html
FAILED: 403 ssa-16.pdf text/html
FAILED: 403 ssa-18.pdf text/html
FAILED: 403 ssa-24.pdf text/html
FAILED: 403 ssa-25.pdf text/html
FAILED: 403 ssa-44.pdf text/html
FAILED: 403 ssa-55.pdf text/html
FAILED: 403 ssa-1-bk.pdf text/html
FAILED: 403 ssa-2-bk.pdf text/html
FAILED: 403 ssa-3.pdf text/html
FAILED: 403 ssa-4.pdf text/html
FAILED: 403 ssa-6.pdf text/html
FAILED: 403 ssa-7.pdf text/html
FAILED: 403 ssa-8.pdf text/html
FAILED: 403 ssa-9.pdf text/html
FAILED: 403 ssa-10-bk.pdf text/html
FAILED: 403 ssa-18-bk.pdf text/html
FAILED: 403 ssa-3368.pdf text/html
FAILED: 403 ssa-3369.pdf text/html
FAILED: 403 ssa-827.pdf text/html
FAILED: 403 ssa-454.pdf text/html
FAILED: 403 ssa-455.pdf text/html
FAILED: 403 ssa-454-bk.pdf text/html
FAILED: 403 ssa-3441.pdf text/html
FAILED: 403 ssa-3820.pdf text/html
FAILED: 403 ssa-3820-bk.pdf text/html
FAILED: 403 ssa-4734.p

In [ ]:
import requests

url = "https://www.ssa.gov/forms/ssa-1.pdf"

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/120.0 Safari/537.36"
    ),
    "Accept": "application/pdf",
    "Referer": "https://www.ssa.gov/forms/"
}

response = requests.get(
    url,
    headers=headers,
    timeout=30
)

print(response.status_code)
print(response.headers)
print(response.content[:20])

403
{'Mime-Version': '1.0', 'Content-Type': 'text/html', 'Content-Length': '390', 'Expires': 'Mon, 13 Jul 2026 18:16:35 GMT', 'Cache-Control': 'max-age=0, no-cache', 'Pragma': 'no-cache', 'Date': 'Mon, 13 Jul 2026 18:16:35 GMT', 'Connection': 'close', 'Server-Timing': 'cdn-cache; desc=HIT, edge; dur=1, ak_p; desc="1783966595504_34900557_98469886_14_7554_4_9_-";dur=1', 'Alt-Svc': 'h3=":443"; ma=93600', 'Strict-Transport-Security': 'max-age=31536000 ; includeSubDomains ; preload'}
b'<HTML><HEAD>\n<TITLE>'


## Import Downloaded SSA Documents
Due to the website's access restrictions, the remaining publicly available forms were downloaded using a browser based approach and stored in the project directory.
Namely, an extension called "Downthemall" was utilized to download the forms.
The remainder of the notebook performs the same validation and metadata generation steps used for the other agencies.

In [ ]:
from pathlib import Path

ssa_folder = Path("/content/drive/MyDrive/newstart_ai/data/raw/ssa")

pdfs = list(ssa_folder.glob("*.pdf"))

print("PDF count:", len(pdfs))

for pdf in pdfs[:10]:
    print(pdf.name)

PDF count: 198
ssa-1021-inst.pdf
ss-5-kor-inst.pdf
ssa-714.pdf
ssa-263.pdf
ssa-770-u4.pdf
ssa-1693.pdf
fss4.pdf
ssa-7-f6-sp.pdf
ssa-3sp.pdf
ssa-2512.pdf


## Verify + Validate Documents
Confirmed that each PDF can be openeded and contains readable PDFs. Then went throguh and checked for small PDF files that could indicate incomplete downloads or invalid documents.

In [ ]:
from pathlib import Path
!pip install PyMuPDF
import fitz  # PyMuPDF

failed = []

for pdf in pdfs:

    try:
        doc = fitz.open(pdf)

        if len(doc) == 0:
            failed.append(pdf.name)

        doc.close()

    except Exception:
        failed.append(pdf.name)

print("Unreadable PDFs:", len(failed))

if failed:
    print(failed[:20])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 53.7 MB/s eta 0:00:00
Unreadable PDFs: 0


In [ ]:
small = []

for pdf in pdfs:

    size_kb = pdf.stat().st_size / 1024

    if size_kb < 10:
        small.append((pdf.name, round(size_kb,2)))

print("Files under 10 KB:", len(small))

small[:20]

Files under 10 KB: 0


[]

### Inspect Sample Documents

Extract text from a small random sample of PDFs to verify that the downloaded forms contain readable machine-readable text suitable for downstream preprocessing.

In [ ]:
import random
import fitz

sample = random.sample(pdfs, 5)

for pdf in sample:

    doc = fitz.open(pdf)

    text = ""

    for page in doc:
        text += page.get_text()

    doc.close()

    print("=" * 80)
    print(pdf.name)
    print(text[:500])

ssa-769-u4.pdf
Form SSA-769 (02-2025) UF 
Discontinue Prior Editions 
Social Security Administration
Page 1 of 2 
OMB No. 0960-0348
Request for Change in Time/Place of Disability Hearing
Name of Claimant
Name of Wage Earner or Self-Employed Person
Social Security Number
Spouse's Name and Social Security Number  
(Complete only if Supplemental Security Income Case) 
(DO NOT WRITE IN THIS SPACE)
Type of Benefit:
Disability
Worker
Widow/Widower
Child
SSI
Disability
Blind
Child
Name of Representative, if any
Repre
ssa-3820.pdf
Form SSA-3820-BK (06-2025) UF 
Discontinue Prior Editions 
Social Security Administration
Page 1 of 14 
OMB No. 0960-0577
Disability Report - Child - SSA-3820-BK                                      
Read All Of This Information Before You Begin Completing This Form                       
This Is Not An Application 
If You Need Help
If you need help with this form, complete as much of it as you can, and your interviewer will help you 
finish it.
How To Complete This 

## Generate Document Metadata

In [ ]:
import pandas as pd
import re
from pathlib import Path

ssa_folder = Path("/content/drive/MyDrive/newstart_ai/data/raw/ssa")

pdfs = sorted(ssa_folder.glob("*.pdf"))


def extract_form_number(filename):
    """
    Extract SSA or SS form number from filename.
    Examples:
        ssa-3820.pdf -> SSA-3820
        ss-5.pdf -> SS-5
    """
    name = filename.lower()

    match = re.search(r"(ssa-\d+[a-z\-]*|ss-\d+[a-z\-]*)", name)

    if match:
        return match.group(1).upper()

    return None


def classify_document_type(filename):

    name = filename.lower()

    if "inst" in name or "instructions" in name:
        return "instructions"

    if "guide" in name:
        return "guide"

    if "worksheet" in name:
        return "worksheet"

    return "form"


metadata = []

for pdf in pdfs:

    metadata.append({
        "filename": pdf.name,
        "filepath": str(pdf),
        "agency": "SSA",
        "form_number": extract_form_number(pdf.name),
        "document_type": classify_document_type(pdf.name)
    })

ssa_df = pd.DataFrame(metadata)

ssa_df.head()

,filename,filepath,agency,form_number,document_type
0,CMS-1763-508C.pdf,/content/drive/MyDrive/newstart_ai/data/raw/ss...,SSA,None,form
1,CMS-L564_SP508.pdf,/content/drive/MyDrive/newstart_ai/data/raw/ss...,SSA,None,form
2,SSA-8-SP.pdf,/content/drive/MyDrive/newstart_ai/data/raw/ss...,SSA,SSA-8-SP,form
3,cms-40b-508c-2025-rev-38.pdf,/content/drive/MyDrive/newstart_ai/data/raw/ss...,SSA,None,form
4,cms-40b-s-508c.pdf,/content/drive/MyDrive/newstart_ai/data/raw/ss...,SSA,None,form


## Export Metadata

In [ ]:
metadata_folder = Path("/content/drive/MyDrive/newstart_ai/data/metadata")
metadata_folder.mkdir(parents=True, exist_ok=True)

output = metadata_folder / "ssa_metadata.csv"

ssa_df.to_csv(output, index=False)

print(output)
print("Rows:", len(ssa_df))

/content/drive/MyDrive/newstart_ai/data/metadata/ssa_metadata.csv
Rows: 198
